# NFCorpus: BGE-Small Plus Jev and Cross-Encoder Reranking

Pipeline:

```text
NFCorpus → BGE-small → top 50 → rerank (Jev or a cross-encoder) → top 10 (nDCG@10)
```

Section 11 compares Jev Score, Jev Choice, Jev Iterative Choice, and two local cross-encoders on the same 100 random test queries.

## 1. Install Dependencies

Make sure the Colab runtime is set to **GPU** before running this cell.

In [ ]:
!pip install onnxruntime-gpu \
    -i https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ \
    -qqq

!pip install fastembed-gpu qdrant-client datasets ranx tqdm beir -qqq

In [ ]:
import onnxruntime as ort

print(ort.get_available_providers())

['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


## 2. Imports and Configuration

In [ ]:
import os
import math
import time
from pathlib import Path

from google.colab import userdata

from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

from beir import util
from beir.datasets.data_loader import GenericDataLoader


# =========================================================
# Secrets
# =========================================================

QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
JEV_API_KEY = userdata.get("JEV")

os.environ["TYPESAFE_API_KEY"] = JEV_API_KEY


# =========================================================
# Configuration
# =========================================================

DATASET = "nfcorpus"
SPLIT = "test"

COLLECTION_NAME = "nfcorpus_bge_small"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

RETRIEVAL_K = 10

DATA_DIR = Path("/content/beir_datasets")

# GPU
FASTEMBED_PROVIDERS = ["CUDAExecutionProvider"]


print("Configuration loaded.")

Configuration loaded.


## 3. One-Time Ingestion (GPU)

**Run this section only once** to create and populate the Qdrant collection.

It downloads NFCorpus, embeds the corpus with BGE-small on the GPU, and uploads the vectors and document payloads to Qdrant.

If the collection already exists and you want to keep it, **do not run the ingestion function** because it recreates the collection.

In [ ]:
from tqdm.auto import tqdm


def ingest_nfcorpus():
    """
    One-time NFCorpus ingestion.

    - Downloads NFCorpus
    - Loads the full corpus
    - Embeds documents with BGE-small on GPU
    - Creates the Qdrant collection
    - Uploads vectors + document payloads

    Run this once.
    """

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    # =====================================================
    # Download NFCorpus
    # =====================================================

    dataset_path = util.download_and_unzip(
        f"https://public.ukp.informatik.tu-darmstadt.de/"
        f"thakur/BEIR/datasets/{DATASET}.zip",
        str(DATA_DIR),
    )

    print(f"Dataset path: {dataset_path}")

    # =====================================================
    # Load corpus
    # =====================================================

    corpus, queries, qrels = GenericDataLoader(
        data_folder=dataset_path
    ).load(split=SPLIT)

    print(f"Corpus: {len(corpus):,} documents")
    print(f"Queries: {len(queries):,}")

    # =====================================================
    # Initialize BGE-small
    # =====================================================

    print("\nLoading BGE-small...")

    embedding_model = TextEmbedding(
        model_name=EMBEDDING_MODEL,
        providers=FASTEMBED_PROVIDERS,
    )

    # Actually verify the requested execution provider
    import onnxruntime as ort

    available_providers = ort.get_available_providers()
    print(f"ONNX Runtime providers: {available_providers}")

    if "CUDAExecutionProvider" not in available_providers:
        raise RuntimeError(
            "CUDAExecutionProvider is not available. "
            "GPU embedding cannot be used."
        )

    print("CUDAExecutionProvider available.")

    # =====================================================
    # Prepare documents
    # =====================================================

    corpus_ids = list(corpus.keys())

    # BEIR-style preprocessing:
    # concatenate title + text before embedding
    documents = [
        (
            corpus[doc_id].get("title", "")
            + " "
            + corpus[doc_id].get("text", "")
        ).strip()
        for doc_id in corpus_ids
    ]

    # Keep the original fields in the Qdrant payload
    payloads = [
        {
            "doc_id": doc_id,
            "title": corpus[doc_id].get("title", ""),
            "text": corpus[doc_id].get("text", ""),
        }
        for doc_id in corpus_ids
    ]

    # =====================================================
    # Determine vector dimension
    # =====================================================

    first_vector = next(
        embedding_model.embed(
            [documents[0]],
            batch_size=1,
        )
    )

    vector_size = len(first_vector)

    print(f"Embedding dimension: {vector_size}")

    # =====================================================
    # Connect to Qdrant
    # =====================================================

    client = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
    )

    # =====================================================
    # Recreate collection
    # =====================================================

    if client.collection_exists(COLLECTION_NAME):
        print(f"\nCollection '{COLLECTION_NAME}' already exists.")
        print("Deleting existing collection...")
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )

    print(f"Created collection: {COLLECTION_NAME}")

    # =====================================================
    # Embedding
    # =====================================================

    EMBED_BATCH_SIZE = 256

    num_documents = len(documents)
    num_batches = (
        num_documents + EMBED_BATCH_SIZE - 1
    ) // EMBED_BATCH_SIZE

    print(
        f"\nEmbedding {num_documents:,} documents "
        f"in {num_batches:,} batches..."
    )

    start = time.perf_counter()

    vectors = []

    for batch_idx in tqdm(
        range(num_batches),
        desc="Embedding",
        unit="batch",
    ):
        start_idx = batch_idx * EMBED_BATCH_SIZE
        end_idx = min(
            start_idx + EMBED_BATCH_SIZE,
            num_documents,
        )

        batch_documents = documents[start_idx:end_idx]

        batch_vectors = list(
            embedding_model.embed(
                batch_documents,
                batch_size=EMBED_BATCH_SIZE,
            )
        )

        vectors.extend(batch_vectors)

    embedding_time = time.perf_counter() - start

    print(f"Embedding completed in {embedding_time:.2f}s")

    # =====================================================
    # Upload to Qdrant
    # =====================================================

    print("\nUploading vectors to Qdrant...")

    start = time.perf_counter()

    client.upload_collection(
        collection_name=COLLECTION_NAME,
        ids=list(range(len(corpus_ids))),
        vectors=vectors,
        payload=payloads,
        batch_size=256,
        parallel=2,
        max_retries=3,
    )

    upload_time = time.perf_counter() - start

    print(f"Upload completed in {upload_time:.2f}s")

    # =====================================================
    # Summary
    # =====================================================

    print("\n" + "=" * 60)
    print("INGESTION COMPLETE")
    print("=" * 60)
    print(f"Collection:         {COLLECTION_NAME}")
    print(f"Documents:          {len(corpus_ids):,}")
    print(f"Vector dimension:   {vector_size}")
    print(f"Embedding time:     {embedding_time:.2f}s")
    print(f"Upload time:        {upload_time:.2f}s")
    print(f"Total:              {embedding_time + upload_time:.2f}s")

    return client, embedding_model, corpus, queries, qrels

In [ ]:
client, embedding_model, corpus, queries, qrels = ingest_nfcorpus()

Dataset path: /content/beir_datasets/nfcorpus


Corpus: 3,633 documents
Queries: 323

Loading BGE-small...


ONNX Runtime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
CUDAExecutionProvider available.
Embedding dimension: 384
Created collection: nfcorpus_bge_small

Embedding 3,633 documents in 15 batches...


Embedding completed in 11.39s

Uploading vectors to Qdrant...
Upload completed in 5.88s

INGESTION COMPLETE
Collection:         nfcorpus_bge_small
Documents:          3,633
Vector dimension:   384
Embedding time:     11.39s
Upload time:        5.88s
Total:              17.27s


## 4. Load an Already-Ingested Collection

For subsequent experiments or a new Colab session, use this instead of running the ingestion cell again.

In [ ]:
def load_experiment():
    """
    Load the existing Qdrant collection and NFCorpus data.

    Does NOT modify the Qdrant collection.
    """

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    dataset_path = DATA_DIR / DATASET

    if not dataset_path.exists():
        dataset_path = util.download_and_unzip(
            f"https://public.ukp.informatik.tu-darmstadt.de/"
            f"thakur/BEIR/datasets/{DATASET}.zip",
            str(DATA_DIR),
        )

    corpus, queries, qrels = GenericDataLoader(
        data_folder=dataset_path
    ).load(split=SPLIT)

    client = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
    )

    embedding_model = TextEmbedding(
        model_name=EMBEDDING_MODEL,
        providers=FASTEMBED_PROVIDERS,
    )

    return client, embedding_model, corpus, queries, qrels

In [ ]:
# Use this instead of ingestion when the collection already exists.
client, embedding_model, corpus, queries, qrels = load_experiment()

print(f"Corpus:  {len(corpus):,}")
print(f"Queries: {len(queries):,}")

Corpus:  3,633
Queries: 323


## 5. BGE-Small Retrieval

In [ ]:
def retrieve_top_k(
    client,
    embedding_model,
    query,
    k=10,
):
    """
    Embed the query with BGE-small and retrieve
    the top-k candidates from Qdrant.
    """

    query_vector = list(
        embedding_model.embed([query])
    )[0]

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=k,
        with_payload=True,
    ).points

    return results

## 6. Jev Score Configuration

In [ ]:
!pip install typesafe_sdk

In [ ]:
from typesafe_sdk import TypeSafeClient, Score

jev = TypeSafeClient()

In [ ]:
JEV_CRITERIA = [
    (
        "Not relevant: "
        "the document does not provide information that helps answer "
        "or address the query."
    ),
    (
        "Somewhat relevant: "
        "the document contains information related to the query, "
        "but only partially addresses what the query asks."
    ),
    (
        "Highly relevant: "
        "the document directly provides information that answers "
        "or substantially addresses the query."
    ),
]

## 7. Jev Reranking

Jev reranks the candidates BGE-small retrieved and changes only their order. The single-query example in section 8 uses the top 10, and the comparison in section 11 uses the top 50.

In [ ]:
def jev_score_rerank(query, candidates):
    """
    Score each candidate independently with Jev,
    then sort the candidates by Jev's Score value.

    The candidate set is exactly the same as the
    BGE-small top-k candidate set.
    """

    questions = {}

    for i, candidate in enumerate(candidates):
        payload = candidate.payload

        title = payload.get("title", "")
        text = payload.get("text", "")

        questions[f"candidate_{i}"] = Score(
            instructions=(
                "Assess how relevant the candidate document "
                "is to the user's query.\n\n"
                f"USER QUERY:\n{query}\n\n"
                f"CANDIDATE DOCUMENT:\n"
                f"Title: {title}\n"
                f"Text: {text}"
            ),
            criteria=JEV_CRITERIA,
        )

    response = jev.system_one(
        model="jev-latest",
        state=(
            "Rank documents according to their relevance "
            "to the user's query."
        ),
        questions=questions,
    )

    reranked = []

    for i, candidate in enumerate(candidates):
        answer = response.scores[f"candidate_{i}"]

        reranked.append({
            "candidate": candidate,
            "jev_score": answer.score,
            "jev_confidence": answer.confidence,
            "jev_probabilities": answer.probabilities,
        })

    # Sort by Score itself, NOT confidence.
    reranked.sort(
        key=lambda x: x["jev_score"],
        reverse=True,
    )

    return reranked

## 8. nDCG@10 on One Example

In [ ]:
from ranx import Run, Qrels, evaluate


# =====================================================
# Select ONE evaluation query
# =====================================================

query_id = next(iter(qrels))
query_text = queries[query_id]

print(f"Query ID: {query_id}")
print(f"Query: {query_text}")

# Ground-truth qrels for this query
query_qrels = {
    query_id: qrels[query_id]
}

qrels_single = Qrels(query_qrels)


# =====================================================
# 1. BGE-small baseline
# =====================================================

query_vector = list(
    embedding_model.embed([query_text])
)[0]

bge_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=30,
    with_payload=True,
).points

# Top 10 BGE results
bge_top10 = bge_results[:10]

bge_run = Run({
    query_id: {
        point.payload["doc_id"]: point.score
        for point in bge_top10
    }
})

bge_ndcg = evaluate(
    qrels_single,
    bge_run,
    "ndcg@10",
)

print(f"\nBGE-small nDCG@10: {bge_ndcg:.4f}")


# =====================================================
# 2. Jev reranking
# =====================================================

jev_results = jev_score_rerank(
    query_text,
    bge_results,       # exactly the BGE top-30 candidates
)

jev_run = Run({
    query_id: {
        item["candidate"].payload["doc_id"]: item["jev_score"]
        for item in jev_results[:10]
    }
})

jev_ndcg = evaluate(
    qrels_single,
    jev_run,
    "ndcg@10",
)

print(f"Jev reranked nDCG@10: {jev_ndcg:.4f}")


# =====================================================
# Show results
# =====================================================

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
print(f"BGE-small       : {bge_ndcg:.4f}")
print(f"BGE → Jev       : {jev_ndcg:.4f}")
print(f"Delta            : {jev_ndcg - bge_ndcg:+.4f}")

print("\nJev ranking:")
for rank, item in enumerate(jev_results[:10], start=1):
    candidate = item["candidate"]
    print(
        f"{rank:2d}. "
        f"Jev={item['jev_score']:.3f} "
        f"doc={candidate.payload['doc_id']} "
        f"title={candidate.payload.get('title', '')[:80]}"
    )

Query ID: PLAIN-2
Query: Do Cholesterol Statin Drugs Cause Breast Cancer?



BGE-small nDCG@10: 0.7776
Jev reranked nDCG@10: 0.8162

RESULTS
BGE-small       : 0.7776
BGE → Jev       : 0.8162
Delta            : +0.0386

Jev ranking:
 1. Jev=2.000 doc=MED-2429 title=Statin use and risk of breast cancer: a meta-analysis of observational studies.
 2. Jev=1.990 doc=MED-2431 title=Long-term statin use and risk of ductal and lobular breast cancer among women 55
 3. Jev=1.620 doc=MED-1193 title=The effects of lowering LDL cholesterol with statin therapy in people at low ris
 4. Jev=1.010 doc=MED-2428 title=Women and statin use: a women's health advocacy perspective.
 5. Jev=0.970 doc=MED-10 title=Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland
 6. Jev=0.850 doc=MED-2440 title=Total Cholesterol and Cancer Risk in a Large Prospective Study in Korea
 7. Jev=0.830 doc=MED-2434 title=High ACAT1 expression in estrogen receptor negative basal-like breast cancer cel
 8. Jev=0.810 doc=MED-14 title=Statin use after diagnosis of breast cancer and su

## 9. Other Ways to Rerank with Jev

### 9.1 Iterative Choice

Each round, Jev picks the best remaining candidate with a Choice question, and that candidate takes the next position.

In [ ]:
from typesafe_sdk import Choice


def jev_choice_iterative_rerank(query, candidates, top_k=10):
    """
    Iteratively select the most relevant candidate using Jev Choice.

    At each round, Jev sees the query and all remaining candidates
    and chooses exactly one. The selected candidate is removed before
    the next round.
    """

    reranked = []
    remaining = list(candidates)

    for rank in range(top_k):
        if not remaining:
            break

        criteria = {}

        for i, candidate in enumerate(remaining):
            payload = candidate.payload
            title = payload.get("title", "")
            text = payload.get("text", "")

            criteria[f"candidate_{i}"] = (
                f"Title: {title}\n"
                f"Text: {text}"
            )

        response = jev.system_one(
            model="jev-latest",
            state=(
                f"USER QUERY:\n{query}\n\n"
                "Choose the candidate document that is most relevant "
                "to answering the query."
            ),
            questions={
                "rerank": Choice(
                    instructions=(
                        "Which candidate document is most relevant "
                        "to the user's query?"
                    ),
                    criteria=criteria,
                )
            },
        )

        answer = response.answers["rerank"]
        selected_key = answer.choice

        # Recover selected candidate
        selected_index = int(selected_key.split("_")[1])
        selected_candidate = remaining[selected_index]

        reranked.append({
            "candidate": selected_candidate,
            "jev_choice": selected_key,
            "jev_confidence": answer.confidence,
            "jev_probabilities": answer.probabilities,
        })

        # Remove selected candidate
        remaining.pop(selected_index)

    return reranked


In [ ]:
# =====================================================
# 3. Jev Choice reranking
# =====================================================

jev_choice_results = jev_choice_iterative_rerank(
    query_text,
    bge_results,       # exactly the BGE top-30 candidates
    top_k=10,
)

choice_run = Run({
    query_id: {
        item["candidate"].payload["doc_id"]: 10 - rank
        for rank, item in enumerate(jev_choice_results, start=1)
    }
})

choice_ndcg = evaluate(
    qrels_single,
    choice_run,
    "ndcg@10",
)

print(f"Jev Choice reranked nDCG@10: {choice_ndcg:.4f}")


# =====================================================
# Show results
# =====================================================

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
print(f"BGE-small       : {bge_ndcg:.4f}")
print(f"BGE → Jev Score : {jev_ndcg:.4f}")
print(f"BGE → Jev Choice: {choice_ndcg:.4f}")

print(f"\nScore Delta  : {jev_ndcg - bge_ndcg:+.4f}")
print(f"Choice Delta : {choice_ndcg - bge_ndcg:+.4f}")

print("\nJev Choice ranking:")
for rank, item in enumerate(jev_choice_results, start=1):
    candidate = item["candidate"]
    print(
        f"{rank:2d}. "
        f"confidence={item['jev_confidence']:.3f} "
        f"doc={candidate.payload['doc_id']} "
        f"title={candidate.payload.get('title', '')[:80]}"
    )


Jev Choice reranked nDCG@10: 0.8283

RESULTS
BGE-small       : 0.7776
BGE → Jev Score : 0.8162
BGE → Jev Choice: 0.8283

Score Delta  : +0.0386
Choice Delta : +0.0507

Jev Choice ranking:
 1. confidence=0.460 doc=MED-2429 title=Statin use and risk of breast cancer: a meta-analysis of observational studies.
 2. confidence=0.780 doc=MED-2431 title=Long-term statin use and risk of ductal and lobular breast cancer among women 55
 3. confidence=0.860 doc=MED-1193 title=The effects of lowering LDL cholesterol with statin therapy in people at low ris
 4. confidence=0.390 doc=MED-10 title=Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland
 5. confidence=0.480 doc=MED-2428 title=Women and statin use: a women's health advocacy perspective.
 6. confidence=0.480 doc=MED-14 title=Statin use after diagnosis of breast cancer and survival: a population-based coh
 7. confidence=0.650 doc=MED-2440 title=Total Cholesterol and Cancer Risk in a Large Prospective Study in Korea
 8

### 9.2 Choice With Probabilities

One Choice question over all candidates. The candidates are ranked by Jev's probability for each option.

In [ ]:
from typesafe_sdk import Choice


def jev_choice_rerank(query, candidates):
    """
    Ask Jev to choose the most relevant candidate from the full pool.

    The probability assigned to each candidate is used as the
    reranking score.
    """

    criteria = {}

    for i, candidate in enumerate(candidates):
        payload = candidate.payload

        criteria[f"candidate_{i}"] = (
            f"Title: {payload.get('title', '')}\n"
            f"Text: {payload.get('text', '')}"
        )

    response = jev.system_one(
        model="jev-latest",
        state=(
            f"USER QUERY:\n{query}\n\n"
            "Choose the candidate document that is most relevant "
            "to answering the query."
        ),
        questions={
            "rerank": Choice(
                instructions=(
                    "Which candidate document is most relevant "
                    "to the user's query?"
                ),
                criteria=criteria,
            )
        },
    )

    answer = response.answers["rerank"]

    reranked = [
        {
            "candidate": candidate,
            "jev_probability": answer.probabilities[f"candidate_{i}"],
        }
        for i, candidate in enumerate(candidates)
    ]

    reranked.sort(
        key=lambda x: x["jev_probability"],
        reverse=True,
    )

    return reranked, answer.choice, answer.confidence

In [ ]:
# =====================================================
# 3. Jev Choice reranking
# =====================================================

jev_choice_results, choice_winner, choice_confidence = jev_choice_rerank(
    query_text,
    bge_results,
)

choice_run = Run({
    query_id: {
        item["candidate"].payload["doc_id"]: item["jev_probability"]
        for item in jev_choice_results[:10]
    }
})

choice_ndcg = evaluate(
    qrels_single,
    choice_run,
    "ndcg@10",
)


# =====================================================
# Results
# =====================================================

print(f"\nJev Choice reranked nDCG@10: {choice_ndcg:.4f}")

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)

print(f"BGE-small       : {bge_ndcg:.4f}")
print(f"BGE → Jev Score : {jev_ndcg:.4f}")
print(f"BGE → Jev Choice: {choice_ndcg:.4f}")

print(f"\nScore Delta  : {jev_ndcg - bge_ndcg:+.4f}")
print(f"Choice Delta : {choice_ndcg - bge_ndcg:+.4f}")

print("\nJev Choice ranking:")
for rank, item in enumerate(jev_choice_results[:10], start=1):
    candidate = item["candidate"]

    print(
        f"{rank:2d}. "
        f"prob={item['jev_probability']:.3f} "
        f"doc={candidate.payload['doc_id']} "
        f"title={candidate.payload.get('title', '')[:80]}"
    )

print(f"\nJev Choice winner: {choice_winner}")
print(f"Jev Choice confidence: {choice_confidence:.3f}")


Jev Choice reranked nDCG@10: 0.7436

RESULTS
BGE-small       : 0.7776
BGE → Jev Score : 0.8162
BGE → Jev Choice: 0.7436

Score Delta  : +0.0386
Choice Delta : -0.0340

Jev Choice ranking:
 1. prob=0.510 doc=MED-2429 title=Statin use and risk of breast cancer: a meta-analysis of observational studies.
 2. prob=0.440 doc=MED-2431 title=Long-term statin use and risk of ductal and lobular breast cancer among women 55
 3. prob=0.040 doc=MED-1193 title=The effects of lowering LDL cholesterol with statin therapy in people at low ris
 4. prob=0.010 doc=MED-2428 title=Women and statin use: a women's health advocacy perspective.
 5. prob=0.000 doc=MED-10 title=Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland
 6. prob=0.000 doc=MED-14 title=Statin use after diagnosis of breast cancer and survival: a population-based coh
 7. prob=0.000 doc=MED-4827 title=Men with Low Serum Cholesterol Have a Lower Risk of High-Grade Prostate Cancer i
 8. prob=0.000 doc=MED-4830 title=

## 10. Cross-Encoders

Two local cross-encoders from FastEmbed rerank the same candidates: `BAAI/bge-reranker-base` and `Xenova/ms-marco-MiniLM-L-6-v2`.

In [ ]:
from fastembed.rerank.cross_encoder import TextCrossEncoder

CROSS_ENCODER_MODELS = [
    "BAAI/bge-reranker-base",
    "Xenova/ms-marco-MiniLM-L-6-v2",
]


def cross_encoder_rerank(cross_encoder, query, candidates):
    documents = [
        (
            candidate.payload.get("title", "")
            + " "
            + candidate.payload.get("text", "")
        ).strip()
        for candidate in candidates
    ]

    scores = list(cross_encoder.rerank(query, documents))

    reranked = [
        {"candidate": candidate, "cross_encoder_score": float(score)}
        for candidate, score in zip(candidates, scores)
    ]

    reranked.sort(key=lambda x: x["cross_encoder_score"], reverse=True)

    return reranked

## 11. Comparison on 100 Random Queries

Every method reranks the same BGE-small top 50 for 100 random test queries (seed 42), and nDCG@10 is computed on the reranked top 10.

In [ ]:
import random
# =====================================================
# Configuration
# =====================================================
NUM_QUERIES = 100
CANDIDATE_DEPTH = 50
TOP_K = 10
SEED = 42
random.seed(SEED)

query_ids = random.sample(
    list(qrels.keys()),
    NUM_QUERIES,
)

print(f"Selected {len(query_ids)} queries")
print(query_ids[:5])

Selected 100 queries
['PLAIN-634', 'PLAIN-133', 'PLAIN-1557', 'PLAIN-1398', 'PLAIN-1275']


### 11.1 BGE-Small Baseline

In [ ]:
from ranx import Run, Qrels, evaluate
import time

bge_scores = []

start_time = time.perf_counter()

for query_id in query_ids:

    query_text = queries[query_id]

    qrels_query = Qrels({
        query_id: qrels[query_id]
    })

    query_vector = list(
        embedding_model.embed([query_text])
    )[0]

    bge_results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=CANDIDATE_DEPTH,
        with_payload=True,
    ).points

    bge_run = Run({
        query_id: {
            point.payload["doc_id"]: point.score
            for point in bge_results[:TOP_K]
        }
    })

    bge_ndcg = evaluate(
        qrels_query,
        bge_run,
        "ndcg@10",
    )

    bge_scores.append(bge_ndcg)

bge_time = time.perf_counter() - start_time

print(f"BGE completed in {bge_time:.2f}s")
print(f"Mean nDCG@10: {sum(bge_scores) / len(bge_scores):.4f}")

BGE completed in 17.21s
Mean nDCG@10: 0.3095


### 11.2 Jev Score

In [ ]:
jev_score_scores = []

start_time = time.perf_counter()

for query_id in query_ids:

    query_text = queries[query_id]

    qrels_query = Qrels({
        query_id: qrels[query_id]
    })

    query_vector = list(
        embedding_model.embed([query_text])
    )[0]

    bge_results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=CANDIDATE_DEPTH,
        with_payload=True,
    ).points

    jev_results = jev_score_rerank(
        query_text,
        bge_results,
    )

    jev_run = Run({
        query_id: {
            item["candidate"].payload["doc_id"]: item["jev_score"]
            for item in jev_results[:TOP_K]
        }
    })

    jev_ndcg = evaluate(
        qrels_query,
        jev_run,
        "ndcg@10",
    )

    jev_score_scores.append(jev_ndcg)

jev_score_time = time.perf_counter() - start_time

print(f"Jev Score completed in {jev_score_time:.2f}s")
print(f"Mean nDCG@10: {sum(jev_score_scores) / len(jev_score_scores):.4f}")

Jev Score completed in 57.17s
Mean nDCG@10: 0.3836


### 11.3 Jev Choice

In [ ]:
jev_choice_scores = []

start_time = time.perf_counter()

for query_id in query_ids:

    query_text = queries[query_id]

    qrels_query = Qrels({
        query_id: qrels[query_id]
    })

    query_vector = list(
        embedding_model.embed([query_text])
    )[0]

    bge_results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=CANDIDATE_DEPTH,
        with_payload=True,
    ).points

    jev_choice_results, _, _ = jev_choice_rerank(
        query_text,
        bge_results,
    )

    choice_run = Run({
        query_id: {
            item["candidate"].payload["doc_id"]: item["jev_probability"]
            for item in jev_choice_results[:TOP_K]
        }
    })

    choice_ndcg = evaluate(
        qrels_query,
        choice_run,
        "ndcg@10",
    )

    jev_choice_scores.append(choice_ndcg)

jev_choice_time = time.perf_counter() - start_time

print(f"Jev Choice completed in {jev_choice_time:.2f}s")
print(f"Mean nDCG@10: {sum(jev_choice_scores) / len(jev_choice_scores):.4f}")

Jev Choice completed in 49.66s
Mean nDCG@10: 0.3768


### 11.4 Jev Iterative Choice

In [ ]:
jev_iterative_scores = []

start_time = time.perf_counter()

for query_id in query_ids:

    query_text = queries[query_id]

    qrels_query = Qrels({
        query_id: qrels[query_id]
    })

    query_vector = list(
        embedding_model.embed([query_text])
    )[0]

    bge_results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=CANDIDATE_DEPTH,
        with_payload=True,
    ).points

    jev_iterative_results = jev_choice_iterative_rerank(
        query_text,
        bge_results,
        top_k=TOP_K,
    )

    iterative_run = Run({
        query_id: {
            item["candidate"].payload["doc_id"]: TOP_K - rank
            for rank, item in enumerate(
                jev_iterative_results,
                start=1,
            )
        }
    })

    iterative_ndcg = evaluate(
        qrels_query,
        iterative_run,
        "ndcg@10",
    )

    jev_iterative_scores.append(iterative_ndcg)

jev_iterative_time = time.perf_counter() - start_time

print(f"Jev Iterative completed in {jev_iterative_time:.2f}s")
print(f"Mean nDCG@10: {sum(jev_iterative_scores) / len(jev_iterative_scores):.4f}")

Jev Iterative completed in 265.69s
Mean nDCG@10: 0.3884


### 11.5 Cross-Encoders

The cross-encoders ran in a separate CPU session with the same code, collection settings, and 100 queries. In that session, BGE-small alone scores 0.3071 instead of 0.3095, most likely because the Colab run embedded the documents on the GPU and the queries on the CPU, which shifts a few rankings. Cross-encoder lifts are measured against the baseline of their own session.

In [ ]:
# Retrieve the BGE-small candidates once and score the baseline in this session.
candidates = {}
local_bge_scores = []

for query_id in query_ids:
    query_vector = list(embedding_model.embed([queries[query_id]]))[0]

    candidates[query_id] = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=CANDIDATE_DEPTH,
        with_payload=True,
    ).points

    bge_run = Run({
        query_id: {
            point.payload["doc_id"]: point.score
            for point in candidates[query_id][:TOP_K]
        }
    })

    local_bge_scores.append(
        evaluate(Qrels({query_id: qrels[query_id]}), bge_run, "ndcg@10")
    )

local_bge_mean = sum(local_bge_scores) / len(local_bge_scores)
print(f"BGE-small in this session: mean nDCG@10: {local_bge_mean:.4f}")

# Rerank the same candidates with each cross-encoder.
cross_encoder_results = {}

for model_name in CROSS_ENCODER_MODELS:
    cross_encoder = TextCrossEncoder(model_name=model_name, batch_size=32)
    scores = []
    start_time = time.perf_counter()

    for query_id in query_ids:
        reranked = cross_encoder_rerank(
            cross_encoder,
            queries[query_id],
            candidates[query_id],
        )

        cross_encoder_run = Run({
            query_id: {
                str(item["candidate"].payload["doc_id"]): item["cross_encoder_score"]
                for item in reranked[:TOP_K]
            }
        })

        scores.append(
            evaluate(Qrels({query_id: qrels[query_id]}), cross_encoder_run, "ndcg@10")
        )

    elapsed = time.perf_counter() - start_time
    cross_encoder_results[model_name] = {"scores": scores, "time": elapsed}

    print(
        f"{model_name}: completed in {elapsed:.2f}s, "
        f"mean nDCG@10: {sum(scores) / len(scores):.4f}"
    )

BGE-small in this session: mean nDCG@10: 0.3071
BAAI/bge-reranker-base: completed in 331.60s, mean nDCG@10: 0.2944
Xenova/ms-marco-MiniLM-L-6-v2: completed in 70.79s, mean nDCG@10: 0.3329


## 12. Summary

Jev results, from the Colab run:

In [ ]:
bge_mean = sum(bge_scores) / len(bge_scores)
jev_score_mean = sum(jev_score_scores) / len(jev_score_scores)
jev_choice_mean = sum(jev_choice_scores) / len(jev_choice_scores)
jev_iterative_mean = sum(jev_iterative_scores) / len(jev_iterative_scores)

print("=" * 60)
print(f"NFCorpus — {len(query_ids)} queries")
print("=" * 60)

print(f"BGE-small       : {bge_mean:.4f}  | {bge_time:.1f}s")
print(f"Jev Score       : {jev_score_mean:.4f}  | {jev_score_time:.1f}s")
print(f"Jev Choice      : {jev_choice_mean:.4f}  | {jev_choice_time:.1f}s")
print(f"Jev Iterative   : {jev_iterative_mean:.4f}  | {jev_iterative_time:.1f}s")

print()
print("Δ nDCG@10 vs BGE:")
print(f"Jev Score       : {jev_score_mean - bge_mean:+.4f}")
print(f"Jev Choice      : {jev_choice_mean - bge_mean:+.4f}")
print(f"Jev Iterative   : {jev_iterative_mean - bge_mean:+.4f}")

NFCorpus — 100 queries
BGE-small       : 0.3095  | 17.2s
Jev Score       : 0.3836  | 57.2s
Jev Choice      : 0.3768  | 49.7s
Jev Iterative   : 0.3884  | 265.7s

Δ nDCG@10 vs BGE:
Jev Score       : +0.0741
Jev Choice      : +0.0673
Jev Iterative   : +0.0789


Cross-encoder results, from the CPU session. Compare the lifts, not the absolute scores, across the two runs.

In [ ]:
print("Cross-encoders, CPU session, 100 queries, top 50 candidates")
print(f"BGE-small (same session): {local_bge_mean:.4f}")
print()
print("Δ nDCG@10 vs BGE-small in the same session:")

for model_name, result in cross_encoder_results.items():
    mean = sum(result["scores"]) / len(result["scores"])
    print(f"{model_name:30}: {mean:.4f}  ({mean - local_bge_mean:+.4f})  | {result['time']:.1f}s")

Cross-encoders, CPU session, 100 queries, top 50 candidates
BGE-small (same session): 0.3071

Δ nDCG@10 vs BGE-small in the same session:
BAAI/bge-reranker-base        : 0.2944  (-0.0127)  | 331.6s
Xenova/ms-marco-MiniLM-L-6-v2 : 0.3329  (+0.0258)  | 70.8s
